# Training and evaluating Llama 3.2B

In [ ]:
import os
import re
import math
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
from peft import LoraConfig, PeftModel
from datetime import datetime
import json
from trl import SFTConfig

colab = False

if colab:
    from google.colab import drive
    drive.mount('/content/drive')
    data_path = Path("/content/drive/MyDrive/Colab Notebooks/3H1L/data")
    models_path = Path("/content/drive/MyDrive/Colab Notebooks/3H1L/models/Llama")
else:
    data_path = "../data"
    models_path = "../models/Llama"

with open(f"{data_path}/train_nn_set.json", "r") as f:
    train_nn_set = json.load(f)

with open(f"{data_path}/validate_set.json", "r") as f:
    validate_set = json.load(f)

with open(f"{data_path}/test_set.json", "r") as f:
    test_set = json.load(f)

f:\Mike\Documents\Backup\Python\fake-headline-predictor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Transform data into appropriate format

In [16]:
def prompt_transform(target_set, test):

    modified_set = []

    for item in target_set:
        headlines = "\n\n".join([f"{i + 1}: {headline}" for i, headline in enumerate(item["headlines"])])
        headlines = f"Identify which of the four following headlines is fake:\n\n{headlines}\n\nAnswer: "
        if test:
            entry = {
                "date": item["date"],
                "prompt": headlines,
                "answer": item["answer"],
            }
        else:
            entry = {
                "date": item["date"],
                "prompt": headlines,
                "answer": item["answer"],
                "completion": str(item["answer"] + 1)
            }

        modified_set.append(entry)

    return modified_set

In [17]:
train_set_Llama_ready = prompt_transform(train_nn_set, False)
validate_set_Llama_ready = prompt_transform(validate_set, False)
test_set_Llama_ready = prompt_transform(test_set, True)

### Model config

Quantisation

In [22]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"

if colab:
    capability = torch.cuda.get_device_capability()
    use_bf16 = capability[0] >= 8
else:
    use_bf16 = False

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )

In [23]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

f:\Mike\Documents\Backup\Python\fake-headline-predictor\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\popro\.cache\huggingface\hub\models--meta-llama--Llama-3.2-3B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 254/254 [00:31<00:00,  7.97it/s]


Memory footprint: 2197.6 MB


Parameter definition, assignment and storage

In [ ]:
# LoRA parameters

LORA_R = 32
LORA_ALPHA = LORA_R * 2
LORA_DROPOUT = 0.1

ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
TARGET_MODULES = ATTENTION_LAYERS

LORA_BIAS = "none"
LORA_TASK_TYPE = "CAUSAL_LM"


# Training parameters

EPOCHS = 3
BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = "cosine"

MAX_GRAD_NORM = 0.3
MAX_SEQUENCE_LENGTH = 256

SAVE_STRATEGY = "epoch"
SAVE_TOTAL_LIMIT = 3

EVAL_STRATEGY = "epoch"

LOGGING_STEPS = 1
REPORT_TO = "none"

LOAD_BEST_MODEL_AT_END = True
METRIC_FOR_BEST_MODEL = "eval_loss"
GREATER_IS_BETTER = False

GROUP_BY_LENGTH = True

# Assign LoRA configuration

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias=LORA_BIAS,
    task_type=LORA_TASK_TYPE,
    target_modules=TARGET_MODULES,
)

# Assign training args

output_dir = str(run_dir / "checkpoints")

train_parameters = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_strategy=SAVE_STRATEGY,
    save_total_limit=SAVE_TOTAL_LIMIT,
    logging_steps=LOGGING_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=MAX_GRAD_NORM,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=GROUP_BY_LENGTH,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to=REPORT_TO,
    max_length=MAX_SEQUENCE_LENGTH,
    eval_strategy=EVAL_STRATEGY,
    load_best_model_at_end=LOAD_BEST_MODEL_AT_END,
    metric_for_best_model=METRIC_FOR_BEST_MODEL,
    greater_is_better=GREATER_IS_BETTER,
)

